# 6.25 Data augmentation

Augmentation teaches invariance by showing many label-preserving views of the same example.

The augmented-risk objective trains on transformed examples whose labels should remain unchanged. Save a copy to Drive to edit.

In [ ]:

import math
import time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

np.random.seed(42)


def clf_digits_ladder():
    """D1 XOR -> D2 blobs -> D3 noisy moons -> D4 digits -> D5 noisy digits."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def pad_to_64(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 64:
        return X.copy()
    if X.shape[1] > 64:
        return X[:, :64].copy()
    out = np.zeros((X.shape[0], 64), dtype=float)
    out[:, :X.shape[1]] = X
    return out


def split_scale(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te




def softmax_logits(X, W):
    logits = X @ W
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def train_small_mlp_loss(X, y, random_state=0, max_iter=90):
    x_tr, x_te, y_tr, y_te = split_scale(X, y)
    clf = MLPClassifier(hidden_layer_sizes=(16,), activation="relu", solver="adam", alpha=0.001, max_iter=max_iter, random_state=random_state)
    clf.fit(x_tr, y_tr)
    proba = clf.predict_proba(x_te)
    loss = log_loss(y_te, proba, labels=clf.classes_)
    acc = accuracy_score(y_te, clf.predict(x_te))
    return float(loss), float(acc), clf


def train_small_mlp_acc(X, y, random_state=0, max_iter=90):
    loss, acc, clf = train_small_mlp_loss(X, y, random_state=random_state, max_iter=max_iter)
    return acc, loss, clf


def preview_ladder(rungs):
    for rung, (name, X, y) in enumerate(rungs, 1):
        classes = np.unique(y)
        print(f"D{rung}: {name:36s} X={X.shape} classes={len(classes)} sample_y={classes[:5].tolist()}")


def plot_metric_table(rows, metric_name):
    print(f"{'rung':<4} {'dataset':<36} {metric_name:>10}")
    for row in rows:
        print(f"{row['rung']:<4} {row['name']:<36} {row['metric']:10.4f}")


def plot_summary(rows, metric_name, ylabel):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    names = [row["rung"] for row in rows]
    values = [row["metric"] for row in rows]
    axes[0].bar(names, values, color="steelblue")
    axes[0].set_title("Metric by ladder rung")
    axes[0].set_ylabel(ylabel)
    axes[1].plot(names, values, marker="o")
    axes[1].set_title("D1→D5 trend")
    axes[1].set_ylabel(ylabel)
    fig.tight_layout()
    plt.show()


def show_artifacts(rungs, title):
    fig, axes = plt.subplots(1, len(rungs), figsize=(13, 2.6))
    for ax, (name, X, y) in zip(axes, rungs):
        if X.shape[1] == 64:
            ax.imshow(X[0].reshape(8, 8), cmap="gray")
            ax.set_title(name.split()[0])
        else:
            ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="tab10")
            ax.set_title(name.split()[0])
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


## The concept, built once

The augmented risk is $R_{aug}(h)=\mathbb{E}_{(x,y)}\mathbb{E}_{t\sim T}\ell(h(t(x)),y)$. The lesson scalar pass gives score $2.1$ and update $2.0-0.05\cdot1.8=1.91$.

In [ ]:

def jitter_points(X, rng, noise=0.08):
    return X + rng.normal(0.0, noise, size=X.shape)


def shift_image(img, dx, dy):
    out = np.zeros_like(img)
    src_r0 = max(0, -dy)
    src_r1 = min(img.shape[0], img.shape[0] - dy)
    src_c0 = max(0, -dx)
    src_c1 = min(img.shape[1], img.shape[1] - dx)
    dst_r0 = max(0, dy)
    dst_r1 = min(img.shape[0], img.shape[0] + dy)
    dst_c0 = max(0, dx)
    dst_c1 = min(img.shape[1], img.shape[1] + dx)
    out[dst_r0:dst_r1, dst_c0:dst_c1] = img[src_r0:src_r1, src_c0:src_c1]
    return out


def rotate_image(img, angle_degrees):
    angle = np.deg2rad(angle_degrees)
    center = (np.array(img.shape) - 1) / 2.0
    out = np.zeros_like(img)
    rot = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    for r in range(img.shape[0]):
        for c in range(img.shape[1]):
            src = rot.T @ (np.array([r, c]) - center) + center
            rr = int(round(src[0]))
            cc = int(round(src[1]))
            if 0 <= rr < img.shape[0] and 0 <= cc < img.shape[1]:
                out[r, c] = img[rr, cc]
    return out


def augment_images(X, rng, copies=2):
    augmented = [X]
    limit = min(len(X), 80)
    for copy in range(copies):
        imgs = []
        for row in X[:limit]:
            img = row.reshape(8, 8)
            angle = rng.choice([-12, -6, 6, 12])
            dx = int(rng.choice([-1, 0, 1]))
            dy = int(rng.choice([-1, 0, 1]))
            aug = rotate_image(img, angle)
            aug = shift_image(aug, dx, dy)
            aug = np.clip(aug + rng.normal(0.0, 0.04, size=aug.shape), 0.0, 1.0)
            imgs.append(aug.reshape(-1))
        augmented.append(np.array(imgs))
    return np.vstack(augmented)

X = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
y = np.array([0, 0, 1, 1])
rng = np.random.default_rng(25)
Xa = jitter_points(X, rng)
aug_risk = np.mean((Xa[:, 0] - y) ** 2)
score = 1.0 * 1.5 + -0.4 * -0.5 + 0.4
updated = 2.0 - 0.05 * 1.8
print("augmented D1 shape", Xa.shape)
print("illustrative augmented risk", aug_risk)
assert Xa.shape == X.shape
assert np.array_equal(y, np.array([0, 0, 1, 1]))
assert np.isclose(score, 2.1)
assert np.isclose(updated, 1.91)


The reusable pipeline rotates, shifts, and injects noise for 8×8 digit images; low-dimensional rungs use label-preserving jitter.

In [ ]:

def augment_then_train(X, y, use_aug=True):
    X64 = pad_to_64(X)
    x_tr, x_te, y_tr, y_te = split_scale(X64, y)
    rng = np.random.default_rng(26)
    if use_aug and X.shape[1] == 64:
        raw_train = x_tr.reshape(-1, 8, 8).reshape(len(x_tr), -1)
        aug = augment_images(raw_train, rng, copies=1)
        x_train = aug
        extra = len(x_train) - len(x_tr)
        y_train = np.concatenate([y_tr, y_tr[:extra]])
    elif use_aug:
        x_train = np.vstack([x_tr, jitter_points(x_tr, rng, noise=0.08)])
        y_train = np.tile(y_tr, 2)
    else:
        x_train = x_tr
        y_train = y_tr
    clf = LogisticRegression(max_iter=400)
    clf.fit(x_train, y_train)
    return float(accuracy_score(y_te, clf.predict(x_te)))


## The dataset ladder

The same code is run from a four-point XOR problem through real 8×8 digit images and a noisy shifted D5 variant.

In [ ]:
rungs = clf_digits_ladder()
preview_ladder(rungs)
show_artifacts(rungs, 'Ladder preview')

## Run the SAME method across D1–D5

In [ ]:

rows = []
for idx, (name, X, y) in enumerate(rungs, 1):
    acc = augment_then_train(X, y, use_aug=True)
    rows.append({"rung": f"D{idx}", "name": name, "metric": acc})
plot_metric_table(rows, "accuracy")


## Results visualization

In [ ]:
show_artifacts(rungs, 'Augmentation artifacts')
plot_summary(rows, 'accuracy', 'held-out accuracy')

## Pitfall on D5: no-augmentation overfit

In [ ]:

name, X, y = rungs[-1]
plain = augment_then_train(X, y, use_aug=False)
augmented = augment_then_train(X, y, use_aug=True)
print("no augmentation", plain)
print("rotate/shift/noise augmentation", augmented)
assert augmented >= plain - 0.08


## Evaluate it + Practice

- Metric: held-out accuracy; compare it with a majority-class or plain logistic baseline.
- Sanity check: D1 should be hand-checkable before trusting D5.
- Ablation: turn off the key idea and the reported metric should get worse or the pitfall should reappear.
- Failure signals: unstable losses, collapsed routing, inflated gradient error, or a D5 result that ignores label noise.

Practice 1: change one hyperparameter and rerun the D1 assert before touching D5.

Practice 2: add a baseline row to the ladder table and explain the largest gap.

Practice 3: make the D5 pitfall more severe, then show the same fix still helps.